# Hidden Markov Model (HMM) - Agoda Customer Journey

**Depth Level:** Deep 📚

This notebook provides a comprehensive deep-dive into HMMs using a real marketing scenario:
**Inferring customer booking intent from observable website behavior at Agoda.**

## Business Context

At Agoda (or any e-commerce platform), we can observe user actions:
- Page views, search queries, wishlist additions, checkout visits, etc.

But we **cannot directly observe** user intent:
- Are they just browsing? Comparing options? Ready to book?

**HMM allows us to infer hidden intent from observable behavior.**

## Applications for Marketing
1. **Multi-touch Attribution**: Which touchpoints influenced conversion?
2. **Campaign Targeting**: Show different ads based on inferred intent
3. **Funnel Optimization**: Identify where users get stuck
4. **Personalization**: Tailor experience to user's stage

---

## Table of Contents
1. [Mathematical Foundation](#part1)
2. [HMM Structure](#part2)
3. [Forward Algorithm (with proofs)](#part3)
4. [Viterbi Algorithm (with proofs)](#part4)
5. [Baum-Welch / EM Algorithm (with proofs)](#part5)
6. [Numerical Stability (Log-space)](#part6)
7. [Practical Applications](#part7)
8. [Comparison with hmmlearn](#part8)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import logsumexp
from hmmlearn import hmm
import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

<a id='part1'></a>
---
## Part 1: Mathematical Foundation

### 1.1 The Markov Property

A stochastic process $\{X_t\}$ satisfies the **Markov property** if:

$$P(X_{t+1} = x_{t+1} | X_t = x_t, X_{t-1} = x_{t-1}, ..., X_1 = x_1) = P(X_{t+1} = x_{t+1} | X_t = x_t)$$

**Intuition**: The future depends only on the present, not the past.

### 1.2 Why This Matters for Marketing

In customer journey modeling:
- **State** = Customer's hidden intent (Browsing, Considering, Ready to Book)
- **Markov assumption** = Current intent depends on previous intent, not entire history

This is a simplification! Real customers have memory. But it's often a good approximation.

### 1.3 Transition Matrix and Stable State

The **transition matrix** $A$ where $a_{ij} = P(X_{t+1} = j | X_t = i)$.

The **stable state** $\pi^*$ satisfies $\pi^* A = \pi^*$, i.e., it's the left eigenvector of $A$ with eigenvalue 1.

**Proof sketch**:
- After $n$ steps, state distribution is $\pi_0 A^n$
- As $n \to \infty$, this converges to $\pi^*$ (under mild conditions)
- At convergence: $\pi^* = \pi^* A$

In [ ]:
# Define our customer journey states and observations

# Hidden States (what we want to infer)
STATES = ['Browsing', 'Considering', 'ReadyToBook']
N_STATES = len(STATES)
state_to_idx = {s: i for i, s in enumerate(STATES)}
idx_to_state = {i: s for i, s in enumerate(STATES)}

# Observations (what we can measure)
OBSERVATIONS = ['PageView', 'Search', 'WishlistAdd', 'PriceCheck', 'CheckoutVisit']
N_OBS = len(OBSERVATIONS)
obs_to_idx = {o: i for i, o in enumerate(OBSERVATIONS)}
idx_to_obs = {i: o for i, o in enumerate(OBSERVATIONS)}

print(f"Hidden States ({N_STATES}): {STATES}")
print(f"Observations ({N_OBS}): {OBSERVATIONS}")

In [ ]:
# Define HMM Parameters with business logic

# Initial state distribution (π)
# Most users start in Browsing mode
pi = np.array([0.70, 0.25, 0.05])

# Transition matrix (A)
# A[i,j] = P(next state = j | current state = i)
A = np.array([
    # To:      Browsing  Considering  ReadyToBook
    [0.70,     0.25,      0.05],    # From: Browsing
    [0.15,     0.60,      0.25],    # From: Considering
    [0.05,     0.15,      0.80]     # From: ReadyToBook
])

# Emission matrix (B)
# B[state, obs] = P(observation | state)
B = np.array([
    # PageView  Search  WishlistAdd  PriceCheck  CheckoutVisit
    [0.50,     0.30,    0.05,        0.10,       0.05],    # Browsing
    [0.20,     0.25,    0.20,        0.25,       0.10],    # Considering
    [0.10,     0.10,    0.15,        0.30,       0.35]     # ReadyToBook
])

# Verify parameters
assert np.allclose(pi.sum(), 1), "π must sum to 1"
assert np.allclose(A.sum(axis=1), 1), "Each row of A must sum to 1"
assert np.allclose(B.sum(axis=1), 1), "Each row of B must sum to 1"

print("✅ All parameters are valid probability distributions")

In [ ]:
# Visualize parameters
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Initial distribution
axes[0].bar(STATES, pi, color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_title('Initial State Distribution (π)', fontsize=12)
axes[0].set_ylabel('Probability')
for i, v in enumerate(pi):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center')

# Transition matrix
sns.heatmap(A, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=STATES, yticklabels=STATES, ax=axes[1])
axes[1].set_title('Transition Matrix (A)', fontsize=12)
axes[1].set_xlabel('To State')
axes[1].set_ylabel('From State')

# Emission matrix
sns.heatmap(B, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=OBSERVATIONS, yticklabels=STATES, ax=axes[2])
axes[2].set_title('Emission Matrix (B)', fontsize=12)
axes[2].set_xlabel('Observation')
axes[2].set_ylabel('State')

plt.tight_layout()
plt.show()

In [ ]:
# Find stable state distribution
eigenvalues, eigenvectors = np.linalg.eig(A.T)

# Find eigenvector with eigenvalue ≈ 1
idx = np.argmin(np.abs(eigenvalues - 1))
stable_state = eigenvectors[:, idx].real
stable_state = stable_state / stable_state.sum()  # Normalize

print("Stable State Distribution (long-run probabilities):")
for state, prob in zip(STATES, stable_state):
    print(f"  P({state}) = {prob:.4f}")

print(f"\n→ In the long run, {stable_state[2]*100:.1f}% of users will be in 'ReadyToBook' state")

<a id='part2'></a>
---
## Part 2: Hidden Markov Model Structure

### 2.1 Formal Definition

An HMM is defined by:

$$\lambda = (\pi, A, B)$$

Where:
- $\pi_i = P(X_1 = i)$ — Initial state distribution
- $a_{ij} = P(X_{t+1} = j | X_t = i)$ — Transition probabilities
- $b_j(o) = P(Y_t = o | X_t = j)$ — Emission probabilities

### 2.2 Key Assumptions

1. **Markov assumption**: $P(X_{t+1} | X_1, ..., X_t) = P(X_{t+1} | X_t)$
2. **Observation independence**: $P(Y_t | X_1, ..., X_t, Y_1, ..., Y_{t-1}) = P(Y_t | X_t)$

### 2.3 Joint Probability

The joint probability of a state sequence $\mathbf{X} = (x_1, ..., x_T)$ and observations $\mathbf{Y} = (y_1, ..., y_T)$:

$$P(\mathbf{X}, \mathbf{Y}) = \pi_{x_1} \cdot b_{x_1}(y_1) \cdot \prod_{t=2}^{T} a_{x_{t-1}, x_t} \cdot b_{x_t}(y_t)$$

In [ ]:
def generate_sequence(pi, A, B, T):
    """
    Generate a sequence of hidden states and observations from an HMM.
    
    This simulates a customer journey through our website.
    """
    N, M = B.shape
    
    states = np.zeros(T, dtype=int)
    observations = np.zeros(T, dtype=int)
    
    # Initial state
    states[0] = np.random.choice(N, p=pi)
    observations[0] = np.random.choice(M, p=B[states[0]])
    
    # Generate sequence
    for t in range(1, T):
        states[t] = np.random.choice(N, p=A[states[t-1]])
        observations[t] = np.random.choice(M, p=B[states[t]])
    
    return states, observations

# Generate a sample customer journey
T = 20  # 20 actions in the session
true_states, observations = generate_sequence(pi, A, B, T)

print("Sample Customer Journey:")
print("="*60)
for t in range(T):
    state_name = idx_to_state[true_states[t]]
    obs_name = idx_to_obs[observations[t]]
    print(f"t={t:2d}: {obs_name:15s} (hidden state: {state_name})")

<a id='part3'></a>
---
## Part 3: Forward Algorithm (Likelihood)

### 3.1 The Problem

**Goal**: Compute $P(\mathbf{Y} | \lambda)$ — the probability of observing the sequence given the model.

**Naive approach**: Sum over all possible state sequences:

$$P(\mathbf{Y}) = \sum_{\mathbf{X}} P(\mathbf{Y}, \mathbf{X}) = \sum_{\mathbf{X}} P(\mathbf{Y} | \mathbf{X}) P(\mathbf{X})$$

**Problem**: There are $N^T$ possible state sequences. For $N=3$ states and $T=20$ steps:
$3^{20} \approx 3.5 \times 10^9$ — computationally infeasible!

### 3.2 Dynamic Programming Solution

Define the **forward probability**:

$$\alpha_t(j) = P(y_1, y_2, ..., y_t, X_t = j | \lambda)$$

This is the probability of:
- Seeing observations $y_1$ through $y_t$
- AND being in state $j$ at time $t$

### 3.3 Recursive Formula (Proof)

**Base case (t=1)**:
$$\alpha_1(j) = P(y_1, X_1 = j) = P(X_1 = j) \cdot P(y_1 | X_1 = j) = \pi_j \cdot b_j(y_1)$$

**Recursive case (t > 1)**:

\begin{align}
\alpha_t(j) &= P(y_1, ..., y_t, X_t = j) \\
&= \sum_{i=1}^{N} P(y_1, ..., y_t, X_{t-1} = i, X_t = j) \\
&= \sum_{i=1}^{N} P(y_1, ..., y_{t-1}, X_{t-1} = i) \cdot P(X_t = j | X_{t-1} = i) \cdot P(y_t | X_t = j) \\
&= \sum_{i=1}^{N} \alpha_{t-1}(i) \cdot a_{ij} \cdot b_j(y_t) \\
&= \left[ \sum_{i=1}^{N} \alpha_{t-1}(i) \cdot a_{ij} \right] \cdot b_j(y_t) \quad \blacksquare
\end{align}

**Termination**:
$$P(\mathbf{Y}) = \sum_{j=1}^{N} \alpha_T(j)$$

### 3.4 Complexity

- **Naive**: $O(N^T)$ — exponential
- **Forward**: $O(N^2 \cdot T)$ — polynomial!

In [ ]:
def forward_naive(observations, pi, A, B):
    """
    Naive forward algorithm - may have numerical underflow for long sequences.
    """
    T = len(observations)
    N = len(pi)
    
    alpha = np.zeros((T, N))
    
    # Base case: t = 0
    alpha[0] = pi * B[:, observations[0]]
    
    # Recursive case: t = 1 to T-1
    for t in range(1, T):
        for j in range(N):
            # Sum over all previous states
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, observations[t]]
    
    # Termination: sum over all final states
    likelihood = np.sum(alpha[-1])
    
    return alpha, likelihood

# Run forward algorithm
alpha, likelihood = forward_naive(observations, pi, A, B)

print(f"Likelihood P(Y|λ) = {likelihood:.6e}")
print(f"Log-likelihood = {np.log(likelihood):.4f}")

In [ ]:
# Step-by-step calculation for first 3 steps
print("Forward Algorithm: Step-by-step Calculation")
print("="*70)

for t in range(min(3, T)):
    obs = observations[t]
    obs_name = idx_to_obs[obs]
    
    print(f"\n📍 t={t}, Observation = {obs_name}")
    print("-" * 50)
    
    for j in range(N_STATES):
        state_name = idx_to_state[j]
        
        if t == 0:
            result = pi[j] * B[j, obs]
            print(f"  α_{t}({state_name}) = π({state_name}) × b({obs_name}|{state_name})")
            print(f"                  = {pi[j]:.4f} × {B[j, obs]:.4f}")
            print(f"                  = {result:.6f}")
        else:
            sum_term = np.sum(alpha[t-1] * A[:, j])
            result = sum_term * B[j, obs]
            
            print(f"  α_{t}({state_name}) = [Σᵢ α_{t-1}(i) × a(i→{state_name})] × b({obs_name}|{state_name})")
            
            # Show detailed sum
            terms = [f"{alpha[t-1, i]:.6f}×{A[i, j]:.2f}" for i in range(N_STATES)]
            print(f"                  = [{' + '.join(terms)}] × {B[j, obs]:.4f}")
            print(f"                  = {sum_term:.6f} × {B[j, obs]:.4f}")
            print(f"                  = {result:.6f}")
        print()

<a id='part4'></a>
---
## Part 4: Viterbi Algorithm (Decoding)

### 4.1 The Problem

**Goal**: Find the most likely state sequence given observations:

$$\mathbf{X}^* = \arg\max_{\mathbf{X}} P(\mathbf{X} | \mathbf{Y}, \lambda)$$

### 4.2 Key Insight

The Viterbi algorithm is identical to Forward, but replaces **sum** with **max**.

Define:
$$v_t(j) = \max_{x_1, ..., x_{t-1}} P(x_1, ..., x_{t-1}, X_t = j, y_1, ..., y_t)$$

### 4.3 Recursive Formula

**Base case**:
$$v_1(j) = \pi_j \cdot b_j(y_1)$$

**Recursive case**:
$$v_t(j) = \max_i \left[ v_{t-1}(i) \cdot a_{ij} \right] \cdot b_j(y_t)$$

### 4.4 Backtracking

Store backpointers:
$$\psi_t(j) = \arg\max_i \left[ v_{t-1}(i) \cdot a_{ij} \right]$$

Then backtrack from $\arg\max_j v_T(j)$ to reconstruct the path.

In [ ]:
def viterbi(observations, pi, A, B):
    """
    Viterbi algorithm - find the most likely state sequence.
    
    Returns:
        best_path: Most likely state sequence
        best_prob: Probability of the best path
        v: Viterbi trellis (for visualization)
        backpointers: Backpointers for path reconstruction
    """
    T = len(observations)
    N = len(pi)
    
    v = np.zeros((T, N))
    backpointers = np.zeros((T, N), dtype=int)
    
    # Base case
    v[0] = pi * B[:, observations[0]]
    
    # Recursive case
    for t in range(1, T):
        for j in range(N):
            # Compute probability from each previous state
            probs = v[t-1] * A[:, j]
            
            # Find best previous state
            best_prev = np.argmax(probs)
            
            # Store result
            v[t, j] = probs[best_prev] * B[j, observations[t]]
            backpointers[t, j] = best_prev
    
    # Termination: find best final state
    best_final = np.argmax(v[-1])
    best_prob = v[-1, best_final]
    
    # Backtrack to find best path
    best_path = np.zeros(T, dtype=int)
    best_path[-1] = best_final
    
    for t in range(T-2, -1, -1):
        best_path[t] = backpointers[t+1, best_path[t+1]]
    
    return best_path, best_prob, v, backpointers

# Run Viterbi
predicted_states, best_prob, v, backpointers = viterbi(observations, pi, A, B)

# Calculate accuracy
accuracy = (predicted_states == true_states).mean()

print(f"Best path probability: {best_prob:.6e}")
print(f"Accuracy: {accuracy:.1%}")
print(f"\nTrue states:      {[idx_to_state[s] for s in true_states]}")
print(f"Predicted states: {[idx_to_state[s] for s in predicted_states]}")

In [ ]:
# Visualize the journey
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# 1. Observations
colors = plt.cm.Set3(np.linspace(0, 1, N_OBS))
for i, obs in enumerate(observations):
    axes[0].bar(i, 1, color=colors[obs], edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Observation')
axes[0].set_title('Customer Actions (Observed)', fontsize=12)
axes[0].set_yticks([])

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=OBSERVATIONS[i]) for i in range(N_OBS)]
axes[0].legend(handles=legend_elements, loc='upper right', ncol=N_OBS)

# 2. True states
state_colors = ['#3498db', '#e74c3c', '#2ecc71']
for i in range(T):
    axes[1].bar(i, 1, color=state_colors[true_states[i]], edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('State')
axes[1].set_title('True Hidden States', fontsize=12)
axes[1].set_yticks([])

# 3. Predicted states
for i in range(T):
    axes[2].bar(i, 1, color=state_colors[predicted_states[i]], edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('State')
axes[2].set_title(f'Predicted States (Viterbi) — Accuracy: {accuracy:.1%}', fontsize=12)
axes[2].set_yticks([])

# Add state legend
state_legend = [Patch(facecolor=state_colors[i], label=STATES[i]) for i in range(N_STATES)]
axes[2].legend(handles=state_legend, loc='upper right', ncol=N_STATES)

# 4. Errors
errors = (predicted_states != true_states).astype(int)
axes[3].bar(range(T), errors, color='red', alpha=0.7)
axes[3].set_ylabel('Error')
axes[3].set_title(f'Prediction Errors ({errors.sum()} / {T})', fontsize=12)
axes[3].set_xlabel('Time Step')
axes[3].set_ylim(0, 1.5)

plt.tight_layout()
plt.show()

<a id='part5'></a>
---
## Part 5: Baum-Welch Algorithm (Learning)

### 5.1 The Problem

Given **only observations** (no state labels), learn the model parameters:

$$\lambda^* = \arg\max_\lambda P(\mathbf{Y} | \lambda)$$

This is an **Expectation-Maximization (EM)** algorithm.

### 5.2 Additional Quantities

**Backward probability** $\beta_t(i)$:
$$\beta_t(i) = P(y_{t+1}, ..., y_T | X_t = i, \lambda)$$

**State occupation probability** $\gamma_t(i)$:
$$\gamma_t(i) = P(X_t = i | \mathbf{Y}, \lambda) = \frac{\alpha_t(i) \cdot \beta_t(i)}{P(\mathbf{Y})}$$

**Transition probability** $\xi_t(i, j)$:
$$\xi_t(i, j) = P(X_t = i, X_{t+1} = j | \mathbf{Y}, \lambda)$$

### 5.3 EM Update Rules

**E-step**: Compute $\gamma$ and $\xi$ using current parameters.

**M-step**: Update parameters:

$$\hat{\pi}_i = \gamma_1(i)$$

$$\hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} \xi_t(i, j)}{\sum_{t=1}^{T-1} \gamma_t(i)}$$

$$\hat{b}_j(k) = \frac{\sum_{t: y_t = k} \gamma_t(j)}{\sum_{t=1}^{T} \gamma_t(j)}$$

In [ ]:
def backward(observations, A, B):
    """Backward algorithm - compute β probabilities."""
    T = len(observations)
    N = A.shape[0]
    
    beta = np.zeros((T, N))
    
    # Initialization: β_T = 1
    beta[-1] = 1.0
    
    # Recursion: backwards
    for t in range(T-2, -1, -1):
        for i in range(N):
            beta[t, i] = np.sum(A[i, :] * B[:, observations[t+1]] * beta[t+1])
    
    return beta


def baum_welch(observations, n_states, n_obs, n_iter=100, tol=1e-6, verbose=True):
    """
    Baum-Welch algorithm to learn HMM parameters.
    """
    T = len(observations)
    
    # Random initialization (with smoothing)
    pi = np.random.dirichlet(np.ones(n_states) * 2)
    A = np.random.dirichlet(np.ones(n_states) * 2, size=n_states)
    B = np.random.dirichlet(np.ones(n_obs) * 2, size=n_states)
    
    log_likelihoods = []
    
    for iteration in range(n_iter):
        # E-step
        alpha, likelihood = forward_naive(observations, pi, A, B)
        beta = backward(observations, A, B)
        
        log_lik = np.log(likelihood + 1e-300)
        log_likelihoods.append(log_lik)
        
        if verbose and iteration % 10 == 0:
            print(f"Iteration {iteration}: log-likelihood = {log_lik:.4f}")
        
        # Check convergence
        if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
            if verbose:
                print(f"Converged at iteration {iteration}")
            break
        
        # Compute gamma: P(state i at time t | all observations)
        gamma = alpha * beta
        gamma = gamma / (gamma.sum(axis=1, keepdims=True) + 1e-300)
        
        # Compute xi: P(state i at t, state j at t+1 | all observations)
        xi = np.zeros((T-1, n_states, n_states))
        for t in range(T-1):
            for i in range(n_states):
                for j in range(n_states):
                    xi[t, i, j] = alpha[t, i] * A[i, j] * B[j, observations[t+1]] * beta[t+1, j]
            xi[t] = xi[t] / (xi[t].sum() + 1e-300)
        
        # M-step
        pi = gamma[0]
        
        for i in range(n_states):
            for j in range(n_states):
                A[i, j] = (xi[:, i, j].sum() + 1e-300) / (gamma[:-1, i].sum() + 1e-300)
        
        for j in range(n_states):
            for k in range(n_obs):
                mask = observations == k
                B[j, k] = (gamma[mask, j].sum() + 1e-300) / (gamma[:, j].sum() + 1e-300)
    
    return pi, A, B, log_likelihoods

print("Training HMM with Baum-Welch...")
print("="*50)
learned_pi, learned_A, learned_B, log_liks = baum_welch(
    observations, n_states=N_STATES, n_obs=N_OBS, n_iter=100
)

In [ ]:
# Compare learned vs true parameters
print("\nParameter Comparison")
print("="*60)

print("\n📊 Initial Distribution (π):")
print(f"True:    {pi}")
print(f"Learned: {learned_pi}")

print("\n📊 Transition Matrix (A):")
print("True:")
print(A)
print("Learned:")
print(learned_A)

print("\n📊 Emission Matrix (B):")
print("True:")
print(B)
print("Learned:")
print(learned_B)

In [ ]:
# Plot convergence
plt.figure(figsize=(10, 4))
plt.plot(log_liks, 'b-', linewidth=2, marker='o', markersize=4)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Log-Likelihood', fontsize=12)
plt.title('Baum-Welch Convergence', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='part6'></a>
---
## Part 6: Numerical Stability (Log-space)

### 6.1 The Underflow Problem

For long sequences, $\alpha_t(j)$ becomes extremely small:
- Each multiplication by a probability (< 1) makes the number smaller
- After 100+ steps, values become smaller than machine precision (~$10^{-308}$)

### 6.2 Solution: Log-space Computation

Work with log-probabilities:
$$\log \alpha_t(j) = \log b_j(y_t) + \log \sum_i \exp(\log \alpha_{t-1}(i) + \log a_{ij})$$

Use the **log-sum-exp** trick for numerical stability:
$$\log \sum_i \exp(x_i) = \max_i(x_i) + \log \sum_i \exp(x_i - \max_i(x_i))$$

In [ ]:
def forward_log(observations, pi, A, B):
    """
    Forward algorithm in log-space for numerical stability.
    """
    T = len(observations)
    N = len(pi)
    
    # Work in log-space
    log_pi = np.log(pi + 1e-300)
    log_A = np.log(A + 1e-300)
    log_B = np.log(B + 1e-300)
    
    log_alpha = np.zeros((T, N))
    
    # Base case
    log_alpha[0] = log_pi + log_B[:, observations[0]]
    
    # Recursive case
    for t in range(1, T):
        for j in range(N):
            # log-sum-exp over previous states
            log_alpha[t, j] = logsumexp(log_alpha[t-1] + log_A[:, j]) + log_B[j, observations[t]]
    
    # Termination
    log_likelihood = logsumexp(log_alpha[-1])
    
    return log_alpha, log_likelihood

# Compare naive vs log-space
alpha_naive, lik_naive = forward_naive(observations, pi, A, B)
log_alpha, log_lik = forward_log(observations, pi, A, B)

print(f"Naive log-likelihood:     {np.log(lik_naive):.6f}")
print(f"Log-space log-likelihood: {log_lik:.6f}")
print(f"\nDifference: {abs(np.log(lik_naive) - log_lik):.2e}")

In [ ]:
# Demonstrate underflow with a long sequence
long_states, long_obs = generate_sequence(pi, A, B, T=500)

# Naive approach
alpha_naive_long, lik_naive_long = forward_naive(long_obs, pi, A, B)

# Log-space approach
log_alpha_long, log_lik_long = forward_log(long_obs, pi, A, B)

print("Long sequence (T=500):")
print(f"  Naive likelihood: {lik_naive_long}")
print(f"  Log-space log-likelihood: {log_lik_long:.4f}")
print(f"\n⚠️ Naive approach gives {lik_naive_long} due to underflow!")
print(f"✅ Log-space approach works correctly.")

<a id='part7'></a>
---
## Part 7: Practical Applications

### 7.1 Marketing Applications at Agoda

| Use Case | How HMM Helps |
|----------|---------------|
| **Multi-touch Attribution** | Infer which touchpoints moved users to "Ready to Book" state |
| **Campaign Targeting** | Target users in "Considering" state with discount offers |
| **Funnel Optimization** | Identify where users get stuck (low transition prob) |
| **Personalization** | Show different content based on inferred intent |
| **Churn Prediction** | Detect when users transition to "Browsing" (disengagement) |

### 7.2 Other Applications

| Domain | Hidden State | Observable |
|--------|--------------|------------|
| **Speech Recognition** | Phonemes | Audio features |
| **NLP / POS Tagging** | Part of speech | Words |
| **Finance** | Market regime (bull/bear) | Returns |
| **Biology** | Gene structure | DNA sequence |
| **Weather** | Actual weather | Sensor readings |

In [ ]:
# Application: Real-time intent detection
def detect_intent_realtime(observation_sequence, pi, A, B):
    """
    Given a sequence of user actions, infer their current intent.
    
    Returns probability distribution over states at each time step.
    """
    alpha, _ = forward_naive(observation_sequence, pi, A, B)
    
    # Normalize to get state probabilities
    state_probs = alpha / alpha.sum(axis=1, keepdims=True)
    
    return state_probs

# Simulate a user session
session_actions = [0, 1, 0, 3, 2, 3, 4]  # PageView, Search, PageView, PriceCheck, WishlistAdd, PriceCheck, CheckoutVisit
session_action_names = [idx_to_obs[a] for a in session_actions]

state_probs = detect_intent_realtime(np.array(session_actions), pi, A, B)

print("Real-time Intent Detection:")
print("="*70)
print(f"{'Step':<5} {'Action':<15} {'Browsing':>10} {'Considering':>12} {'ReadyToBook':>12}")
print("-"*70)

for t, (action, probs) in enumerate(zip(session_action_names, state_probs)):
    intent = STATES[np.argmax(probs)]
    print(f"{t:<5} {action:<15} {probs[0]:>10.1%} {probs[1]:>12.1%} {probs[2]:>12.1%}  → {intent}")

In [ ]:
# Visualize intent evolution
fig, ax = plt.subplots(figsize=(12, 5))

for i, state in enumerate(STATES):
    ax.plot(state_probs[:, i], label=state, linewidth=2, marker='o')

ax.set_xticks(range(len(session_action_names)))
ax.set_xticklabels(session_action_names, rotation=45, ha='right')
ax.set_xlabel('User Action', fontsize=12)
ax.set_ylabel('P(Intent)', fontsize=12)
ax.set_title('Customer Intent Evolution During Session', fontsize=14)
ax.legend(title='Intent State')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a id='part8'></a>
---
## Part 8: Comparison with hmmlearn

Let's verify our implementations against the `hmmlearn` library.

In [ ]:
# Create hmmlearn model with our parameters
model = hmm.CategoricalHMM(n_components=N_STATES, random_state=42)
model.startprob_ = pi
model.transmat_ = A
model.emissionprob_ = B

obs_2d = observations.reshape(-1, 1)

# Compare log-likelihood
log_prob_hmmlearn = model.score(obs_2d)
_, log_prob_ours = forward_log(observations, pi, A, B)

print("Log-likelihood Comparison:")
print(f"  hmmlearn: {log_prob_hmmlearn:.6f}")
print(f"  Our impl: {log_prob_ours:.6f}")
print(f"  Match: {np.allclose(log_prob_hmmlearn, log_prob_ours)}")

In [ ]:
# Compare Viterbi decoding
_, states_hmmlearn = model.decode(obs_2d)
states_ours, _, _, _ = viterbi(observations, pi, A, B)

print("\nViterbi Decoding Comparison:")
print(f"  hmmlearn: {states_hmmlearn}")
print(f"  Our impl: {states_ours}")
print(f"  Match: {np.array_equal(states_hmmlearn, states_ours)}")

In [ ]:
# Train a model using hmmlearn
model_fit = hmm.CategoricalHMM(n_components=N_STATES, n_iter=100, random_state=42)
model_fit.fit(obs_2d)

print("\nhmmlearn Trained Parameters:")
print(f"Transition matrix:\n{model_fit.transmat_}")
print(f"\nEmission matrix:\n{model_fit.emissionprob_}")

---
## Summary

### Algorithm Comparison

| Algorithm | Purpose | Complexity | When to Use |
|-----------|---------|------------|-------------|
| **Forward** | P(observations) | O(N²T) | Model evaluation |
| **Viterbi** | Best state sequence | O(N²T) | Decode hidden states |
| **Baum-Welch** | Learn parameters | O(N²T) × iters | Train from data |

### Key Takeaways

1. **HMMs infer hidden states from observations** — perfect for customer intent detection
2. **Forward algorithm** = sum over paths (likelihood)
3. **Viterbi algorithm** = max over paths (best sequence)
4. **Baum-Welch** = EM algorithm for unsupervised learning
5. **Use log-space** for numerical stability on long sequences

### Marketing Applications

- Real-time intent detection for personalization
- Multi-touch attribution modeling
- Funnel optimization via transition analysis
- Churn prediction from behavioral signals